# 原始交易数据质量检查

检查原始工作簿的结构，以及 10 月窗口内的缺失、整行重复和异常交易。

In [1]:
import pandas as pd
import numpy as np
data = pd.read_excel('../data/raw/Online Retail.xlsx')

In [ ]:
# 检查全表规模和日期字段
x, y = data.shape
print(f"原数据为{x}行{y}列")
print(f"日期缺失行数为{data['InvoiceDate'].isna().sum()}")
print(f"日期类型为{data['InvoiceDate'].dtypes}")

原数据为541909行8列
日期缺失行数为0
日期类型为datetime64[ns]


In [3]:
# 筛选10月信息
data_10 = data.loc[
    (data["InvoiceDate"] >= pd.Timestamp('2011-10-01')) & 
    (data["InvoiceDate"] < pd.Timestamp('2011-11-01'))
]

# 检查最大日期和最小日期(筛选后日期范围)
date_min = data_10['InvoiceDate'].min()
date_max = data_10['InvoiceDate'].max()
print(f"筛选后日期范围为{date_min}--{date_max}")

# 10月子集行数
x_10 = len(data_10)
print(f"筛选后10月份子集行数为{x_10}")

# 检查筛选信息是否有缺失，类型问题
print(f"转换后缺失行为{data_10['InvoiceDate'].isna().sum()}")
print(f"转换后时间类型为{data_10['InvoiceDate'].dtypes}")

筛选后日期范围为2011-10-02 10:32:00--2011-10-31 17:19:00
筛选后10月份子集行数为60742
转换后缺失行为0
转换后时间类型为datetime64[ns]


In [4]:
# 10月各字段缺失数/比例
for i in data_10.columns:
    n = data_10[i].isna().sum()
    print(f"{i}字段缺失数为{n}，缺失比例是{n/x_10:.3%}")

InvoiceNo字段缺失数为0，缺失比例是0.000%
StockCode字段缺失数为0，缺失比例是0.000%
Description字段缺失数为104，缺失比例是0.171%
Quantity字段缺失数为0，缺失比例是0.000%
InvoiceDate字段缺失数为0，缺失比例是0.000%
UnitPrice字段缺失数为0，缺失比例是0.000%
CustomerID字段缺失数为10047，缺失比例是16.540%
Country字段缺失数为0，缺失比例是0.000%


In [5]:
# 统计整行完全重复情况
print(f"重复组全部行为{data_10.duplicated(keep=False).sum()}")
print(f"首条之外的额外行重复行为{data_10.duplicated(keep='first').sum()}")

# 检查InvoiceNo重复
print(f"InvoiceNo重复行为{data_10['InvoiceNo'].duplicated().sum()}")

# 检查InvoiceNo和StockCode组合重复
print(f"InvoiceNo和StockCode重复行为{data_10[['InvoiceNo','StockCode']].duplicated().sum()}")

# 检查同一InvoiceNo跨Country的情况
country_counts = data_10.groupby('InvoiceNo')['Country'].nunique()
invoice_countrys = country_counts[country_counts>1]
print(f"同一InvoiceNo跨Country的情况有{len(invoice_countrys)}条")

重复组全部行为1482
首条之外的额外行重复行为773
InvoiceNo重复行为58105
InvoiceNo和StockCode重复行为1617
同一InvoiceNo跨Country的情况有0条


In [ ]:
# 输出取消编号标记与Quantity正/零/负的交叉计数
# 交叉检查的 Pandas 实现参考了 AI 辅助建议。
data_main = data_10.copy()
data_main['is_cancel'] = data_main['InvoiceNo'].astype('string').str.startswith('C',na=False)
conditions = [
    data_main['Quantity']>0,
    data_main['Quantity']==0,
    data_main['Quantity']<0
]
choices = ['正','零','负']
data_main['quantity_type'] = np.select(conditions,choices,default='缺失')
cross_count = pd.crosstab(
    data_main['is_cancel'],
    data_main['quantity_type']
)
cross_count = cross_count.reindex(columns=['正','零','负'],fill_value=0)
cross_count = cross_count.rename(columns={
    '正':'正数量',
    '零':'零数量',
    '负':'负数量'
})
cross_count.index.name = '是否取消'
display(cross_count)
# 交叉表合计 60,742 行，与 10 月窗口一致。
# 取消和非取消发票均有负数量，无法仅凭数量符号识别取消。

quantity_type,正数量,零数量,负数量
是否取消,,,
False,59396,0,146
True,0,0,1200


In [7]:
# 检查UnitPrice的零和负数
zero_count = len(data_10[data_10['UnitPrice']==0])
negative_count = len(data_10[data_10['UnitPrice']<0])
print(f"UnitPrice的负数为{negative_count}，零数为{zero_count}")

UnitPrice的负数为0，零数为238


In [8]:
# 结合 CustomerID 缺失情况，解释影响
print(f"CustomerID缺失行为{data_10['CustomerID'].isna().sum()}行")
# CustomerID 缺失限制客户级归属和统计。国家字段完整，可用于国家级交易汇总。

CustomerID缺失行为10047行
